In [13]:
from dotenv import load_dotenv,find_dotenv
from langchain.tools import tool
from langchain.messages import AIMessage, HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda


load_dotenv(find_dotenv())

True

In [10]:
model = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=1.0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
ai_msg = model.invoke(messages)
ai_msg

AIMessage(content=[{'type': 'text', 'text': "J'adore la programmation.", 'extras': {'signature': 'Ev0FCvoFAb4+9vvdtvaPlQtHZvgUukLsf3P8cd0Egmy5b3MZQSV68r0STGOB1mgmSmaGOomyKuY1yZ0Xq+Eak67xVvHEJSJEW/WAV1s07Xznyy48GEM8yk4g4AC/0rwVa29HJeCsTejdtoJCGCSjt5CEd9j0XsbN8NR4NvfegFARoGvsNhiNTycMDpHGqvUMRSM7wS/W7wt7mVlD6FicMV/WK2Lm/ciJLGKJm4BHxE3E8ezCWUermGumzE0ZPlHpCDgM6O294fXqz4oRm2aOrdB06d5d2wO2U4fGcZ9BWnZlZVgP4tX4CB6/5omCrZ2Ujg7OngzTBW46UfcSZnd1RUz7ypX6scPMF3zxW6LDv8ZVQSU95WrZ2/2OSuvZsXeiDzH5T73fN7w7nSy699m335lK9tc7nMeHsY2HwmrlOcEXLuIQMsBGyEfWyDY1masga7PL+aRJMATR10jXSM86icWHnJGW6MYOhOxEgpihReELEitHf7kn0KQC1TDWyO/4OUgaXCO26ZDfDija2gKMzA6HCavYhH97OhnIv3CJYaV38exNpycft6hnFiKIGZL+jT0yM3iGZRpXypdD/Gv8VaXPspN3zn5ZJ/KXigV8MXiKbNCFy6O09EiT2M7NHI4Ei8yw8cndAZIyZ6MjD9uGCegqKuVsj94HQApJMs91yPUATPjhVdsQxQ8FAZduhnBLEBMNSh/GH9oAtvmAIXFqHwHFOmwouu50C+zlKWwimQrQKPfosNVqXGW4wVIEU4YjQzrTvG2+rgcv4rdmY4/T4CzPgKAAntan3mO1PMF1Nm+8CHDkmw8wlVDWTprsUd45PA+vckQT2ZqsUFyKx6Tgbrb3D7izqlN+rm+6g6NbqhMwLZ0YpeB86dfTyoxOIakpVsek/9

In [11]:
template = """
You are an expert data scientist with an expertise in building deep learning models. 
Explain the concept of {concept} in a couple of lines
"""

prompt = PromptTemplate(
    input_variables=["concept"],
    template=template,
)

response = model.invoke(prompt.format(concept="molecular biology"))

In [14]:
# Define prompt template
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an expert product reviewer."),
        ("human", "List the main features of the product {product_name}."),
    ]
)


# Define pros analysis step
def analyze_pros(features):
    pros_template = ChatPromptTemplate.from_messages(
        [
            ("system", "You are an expert product reviewer."),
            (
                "human",
                "Given these features: {features}, list the pros of these features.",
            ),
        ]
    )
    return pros_template.format_prompt(features=features)


# Define cons analysis step
def analyze_cons(features):
    cons_template = ChatPromptTemplate.from_messages(
        [
            ("system", "You are an expert product reviewer."),
            (
                "human",
                "Given these features: {features}, list the cons of these features.",
            ),
        ]
    )
    return cons_template.format_prompt(features=features)


# Combine pros and cons into a final review
def combine_pros_cons(pros, cons):
    return f"Pros:\n{pros}\n\nCons:\n{cons}"


# Simplify branches with LCEL
pros_branch_chain = (
    RunnableLambda(lambda x: analyze_pros(x)) | model | StrOutputParser()
)

cons_branch_chain = (
    RunnableLambda(lambda x: analyze_cons(x)) | model | StrOutputParser()
)

# Create the combined chain using LangChain Expression Language (LCEL)
chain = (
    prompt_template
    | model
    | StrOutputParser()
    | RunnableParallel(branches={"pros": pros_branch_chain, "cons": cons_branch_chain})
    | RunnableLambda(lambda x: combine_pros_cons(x["branches"]["pros"], x["branches"]["cons"]))
)

# Run the chain
result = chain.invoke({"product_name": "MacBook Pro"})

# Output
print(result)

Pros:
Based on the features provided, here are the **pros** of the current MacBook Pro (M3 family) from an expert product reviewer’s perspective:

### 1. Unmatched Performance Efficiency
*   **Unified Memory Architecture (UMA):** By allowing the CPU and GPU to share a single pool of memory, the system eliminates data duplication, resulting in significantly faster performance and lower latency.
*   **Scalable Power:** Whether you are a student (M3) or a high-end 3D animator (M3 Max), the chip lineup offers a specific tier of power tailored to your workload.
*   **Next-Gen Graphics:** Hardware-accelerated ray tracing brings the MacBook Pro up to speed with high-end gaming PCs, making it a viable machine for game developers and 3D artists.

### 2. The Gold Standard Display
*   **Desktop-Quality Visuals:** The Mini-LED technology delivers "true blacks" and a contrast ratio that rivals high-end OLED monitors, essential for color-grading and photo editing.
*   **Ultra-Smooth Interaction:** P